# **Mini Project : Pengambilan dan Pembersihan Data Melalui API** 

### Pengambilan dan Eksplorasi Data TV Series Menggunakan TMDB API



**Nama**  Muhammad Amar Primus Firdaus

**Program**  AI Automation Engineer

**Project**  Pengambilan dan Pembersihan Data Melalui API




## **Tahap 1: Mendapatkan API Key yang akan digunakan** 

Pada tahap ini, kita membutuhkan API Key untuk dapat mengakses data dari The Movie Database (TMDB) API. API Key berfungsi sebagai kredensial untuk mengautentikasi setiap permintaan yang dikirimkan ke API TMDB.

Setelah mendapatkan API Key, key tersebut disimpan dalam file **.env** agar tidak dituliskan secara langsung di dalam kode program.

In [1]:
# Menginstall library yang dibutuhkan dalam mengakses API
!pip install requests python-dotenv --quiet

In [2]:
# Mengimport library yang dibutuhkan dalam mengaskes API
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

# Membaca isi file .env
load_dotenv()  

API_KEY = os.getenv("API_KEY")

if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


## **Tahap 2: Melakukan pemanggilan pada API**

TMDB punya beberapa alamat (disebut endpoint) untuk mengakses jenis data yang berbeda. Kita akan menggunakan endpoint bernama discover/tv untuk mendapatkan daftar serial TV.

Dengan params sebagai berikut :
1. api_key : kartu identitas untuk mengakses API TMDB
2. language : bahasa data yang ingin ditampilkan
3. page : nomor halaman data yang ingin diambil
4. sort_by : cara mengurutkan data, misalnya berdasarkan popularitas

In [3]:
# Meminta data serial televisi dari TMDB API berdasarkan tingkat popularitas, menggunakan API key, 
# bahasa Inggris, dan halaman pertama, kemudian menampilkan status respons serta jumlah data yang diperoleh.
alamat_api = "https://api.themoviedb.org/3/discover/tv"

parameter = {
    "api_key"  : API_KEY,
    "language" : "en-US",
    "page"     : 1,
    "sort_by"  : "popularity.desc"
}

response = requests.get(alamat_api, params=parameter)

print(f"Status Code: {response.status_code}")

hasil = response.json()

print(f"Jumlah TV ditemukan (total_results): {hasil['total_results']}")
print(f"Jumlah TV yang dikirim kali ini    : {len(hasil['results'])}")

Status Code: 200
Jumlah TV ditemukan (total_results): 20001
Jumlah TV yang dikirim kali ini    : 20


In [4]:
# Menampilkan hasil data yang didapat setelah permintaan melalui API 
hasil['total_results']
hasil['results']

[{'adult': False,
  'backdrop_path': '/uc1p1PEbEMpdIHqnU9TESNC6Jhp.jpg',
  'genre_ids': [18],
  'id': 275102,
  'origin_country': ['KR'],
  'original_language': 'ko',
  'original_name': '스캔들',
  'overview': 'In Joseon, a secret proposal ignites hidden desires in a gifted noblewoman, a famed lover and an unsuspecting lady caught in their dangerous game.',
  'popularity': 3400.2893,
  'poster_path': '/4TU0PcHBIl0wPW5B6aVRGExOnHE.jpg',
  'first_air_date': '2026-09-18',
  'softcore': False,
  'name': 'The Scandal',
  'vote_average': 6.273,
  'vote_count': 11},
 {'adult': False,
  'backdrop_path': '/pF0qkRsrHkdYadPWY9AMeFZfcwk.jpg',
  'genre_ids': [10759, 80],
  'id': 108978,
  'origin_country': ['US'],
  'original_language': 'en',
  'original_name': 'Reacher',
  'overview': 'Jack Reacher, a veteran military police investigator, has just recently entered civilian life. Reacher is a drifter, carrying no phone and the barest of essentials as he travels the country and explores the nation he o

In [5]:
# Melakukan inisialiasai pada hasil dan menapilkan data pada index 0 atau raw pertama 
tv = hasil["results"][0]
tv

{'adult': False,
 'backdrop_path': '/uc1p1PEbEMpdIHqnU9TESNC6Jhp.jpg',
 'genre_ids': [18],
 'id': 275102,
 'origin_country': ['KR'],
 'original_language': 'ko',
 'original_name': '스캔들',
 'overview': 'In Joseon, a secret proposal ignites hidden desires in a gifted noblewoman, a famed lover and an unsuspecting lady caught in their dangerous game.',
 'popularity': 3400.2893,
 'poster_path': '/4TU0PcHBIl0wPW5B6aVRGExOnHE.jpg',
 'first_air_date': '2026-09-18',
 'softcore': False,
 'name': 'The Scandal',
 'vote_average': 6.273,
 'vote_count': 11}

In [6]:
# Preview pada data yang telah diload, kemudian hanya menapilkan data yang berisi nama, overview, dan tanggal
print("Nama Serial :", tv["name"])
print("Overview    :", tv["overview"])
print("Tanggal     :", tv["first_air_date"])

Nama Serial : The Scandal
Overview    : In Joseon, a secret proposal ignites hidden desires in a gifted noblewoman, a famed lover and an unsuspecting lady caught in their dangerous game.
Tanggal     : 2026-09-18


## **Tahap 3: Mengubah ke dalam bentuk Class**


Pada tahap ini, kita akan menerapkan konsep *Object-Oriented Programming (OOP)* dengan membungkus proses pengambilan data dari TMDB API ke dalam sebuah class.

Sebelumnya, proses pengambilan data dilakukan secara langsung menggunakan requests. Dengan menggunakan class, proses tersebut dapat dibuat lebih terstruktur, terorganisir, dan dapat digunakan kembali tanpa harus menuliskan kode yang sama berulang kali.

In [7]:
### Membungkus dalam bentuk Class

class TMBDFavTV:

    def __init__(self, api_key):
        self.api_key = api_key
        self.alamat_base = "https://api.themoviedb.org/3"

    def get_tv_data(self, target_rows=1000):
        alamat_api = f"{self.alamat_base}/discover/tv"

        # Misalkan dalam satu halaman terdapat 20 data
        jumlah_halaman = (target_rows + 19) // 20
        all_tv = []

        for page in range(1, jumlah_halaman + 1):

            params = {
                "api_key": self.api_key,
                "language": "en-US",
                "page": page,
                "sort_by": "popularity.desc"
            }

            try:
                response = requests.get(alamat_api, params=params)
                print(f"Page {page} | Status Code: {response.status_code}")

                if response.status_code == 200:
                    data = response.json()
                    all_tv.extend(data["results"])
                else:
                    print("Error:", response.text)
                    break

            except requests.exceptions.RequestException as e:
                print(f"Terjadi kesalahan koneksi pada page {page}: {e}")
                break

        # Mengubah hasil pengambilan data menjadi DataFrame
        df = pd.DataFrame(all_tv)

        print(f"\nTotal data terkumpul: {len(df)}")

        return df

In [8]:
# Membuat objek tmbd dari class TMBDFavTV
tmbd = TMBDFavTV(API_KEY)
# Memanggil dan menyimpan data 
df = tmbd.get_tv_data(1000)

Page 1 | Status Code: 200
Page 2 | Status Code: 200
Page 3 | Status Code: 200
Page 4 | Status Code: 200
Page 5 | Status Code: 200
Page 6 | Status Code: 200
Page 7 | Status Code: 200
Page 8 | Status Code: 200
Page 9 | Status Code: 200
Page 10 | Status Code: 200
Page 11 | Status Code: 200
Page 12 | Status Code: 200
Page 13 | Status Code: 200
Page 14 | Status Code: 200
Page 15 | Status Code: 200
Page 16 | Status Code: 200
Page 17 | Status Code: 200
Page 18 | Status Code: 200
Page 19 | Status Code: 200
Page 20 | Status Code: 200
Page 21 | Status Code: 200
Page 22 | Status Code: 200
Page 23 | Status Code: 200
Page 24 | Status Code: 200
Page 25 | Status Code: 200
Page 26 | Status Code: 200
Page 27 | Status Code: 200
Page 28 | Status Code: 200
Page 29 | Status Code: 200
Page 30 | Status Code: 200
Page 31 | Status Code: 200
Page 32 | Status Code: 200
Page 33 | Status Code: 200
Page 34 | Status Code: 200
Page 35 | Status Code: 200
Page 36 | Status Code: 200
Page 37 | Status Code: 200
Page 38 | 

In [22]:
# Menampilkan data dengan memperlihatkan 10 data pertama paling atas
df.head(3)

,adult,backdrop_path,genre_ids,id,origin_country,original_language,original_name,overview,popularity,poster_path,first_air_date,softcore,name,vote_average,vote_count
0,False,/uc1p1PEbEMpdIHqnU9TESNC6Jhp.jpg,[18],275102,[KR],ko,스캔들,"In Joseon, a secret proposal ignites hidden de...",3400.2893,/4TU0PcHBIl0wPW5B6aVRGExOnHE.jpg,2026-09-18,False,The Scandal,6.273,11
1,False,/pF0qkRsrHkdYadPWY9AMeFZfcwk.jpg,"[10759, 80]",108978,[US],en,Reacher,"Jack Reacher, a veteran military police invest...",867.0873,/f1VCQIG2iCyOookdgOzwtUpwWC0.jpg,2022-02-03,False,Reacher,8.100,3383
2,False,/qFfWFwfaEHzDLWLuttWiYq7Poy2.jpg,[10767],2261,[US],en,The Tonight Show Starring Johnny Carson,The Tonight Show Starring Johnny Carson is a t...,809.6015,/uSvET5YUvHNDIeoCpErrbSmasFb.jpg,1962-10-01,False,The Tonight Show Starring Johnny Carson,7.536,96


In [10]:
# Melihat informasi data 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   adult              1000 non-null   bool   
 1   backdrop_path      959 non-null    str    
 2   genre_ids          1000 non-null   object 
 3   id                 1000 non-null   int64  
 4   origin_country     1000 non-null   object 
 5   original_language  1000 non-null   str    
 6   original_name      1000 non-null   str    
 7   overview           1000 non-null   str    
 8   popularity         1000 non-null   float64
 9   poster_path        986 non-null    str    
 10  first_air_date     1000 non-null   str    
 11  softcore           1000 non-null   bool   
 12  name               1000 non-null   str    
 13  vote_average       1000 non-null   float64
 14  vote_count         1000 non-null   int64  
dtypes: bool(2), float64(2), int64(2), object(2), str(7)
memory usage: 103.6+ KB


Berdasarkan hasil informasi data di atas, dapat dilihat bahwa setiap kolom memiliki tipe data yang berbeda. Selanjutnya, perlu diperiksa apakah tipe data yang digunakan pada setiap kolom sudah sesuai dengan karakteristik datanya. Pada kolom first_air_date, tipe data yang digunakan masih berupa str. Hal ini kurang sesuai karena kolom tersebut berisi informasi berupa tanggal. Oleh karena itu, tipe data pada kolom **first_air_date** perlu diubah menjadi **datetime** agar dapat digunakan untuk analisis data berdasarkan waktu. Proses ini kita dapat lakukan pada tahap *Data Cleaning*.

## **Tahap 4: Membersihkan Data (Data Cleaning)**

Pada tahap ini kita diminta untuk melakukan pembersihan data dengan kriteria pengerjaan yang wajib dilakukan sebagai berikut :
1. Nilai yang kosong
2. Baris yang kembar
3. Tipe data yang tidak sesuai


In [21]:
print("1. Jumlah sel kosong per kolom:")
print(df.isnull().sum())

print("\n2. Jumlah baris yang kembar:")
print(df.duplicated(subset="id").sum())

print("\n3. Tipe data setiap kolom:")
print(df.dtypes)

1. Jumlah sel kosong per kolom:
adult                 0
backdrop_path        41
genre_ids             0
id                    0
origin_country        0
original_language     0
original_name         0
overview              0
popularity            0
poster_path          14
first_air_date        0
softcore              0
name                  0
vote_average          0
vote_count            0
dtype: int64

2. Jumlah baris yang kembar:
67

3. Tipe data setiap kolom:
adult                   bool
backdrop_path            str
genre_ids             object
id                     int64
origin_country        object
original_language        str
original_name            str
overview                 str
popularity           float64
poster_path              str
first_air_date           str
softcore                bool
name                     str
vote_average         float64
vote_count             int64
dtype: object


### **Membaca hasil pemeriksaan**

Berdasarkan hasil pemeriksaan data di atas, terdapat beberapa hal yang perlu diperhatikan sebelum data digunakan untuk tahap analisis.

Pertama, terdapat nilai kosong pada dua kolom, yaitu backdrop_path sebanyak 41 data dan poster_path sebanyak 14 data. Nilai kosong tersebut masih dapat dipertahankan karena kedua kolom tersebut hanya berisi informasi mengenai gambar latar dan poster serial TV. Oleh karena itu, nilai kosong akan diisi dengan keterangan "Tidak tersedia", sehingga baris data tidak perlu dihapus.

Kedua, terdapat 67 baris data yang terindikasi kembar berdasarkan kolom id. Kolom id digunakan sebagai acuan karena setiap serial TV pada TMDB memiliki identitas atau ID yang berbeda. Oleh karena itu, data yang memiliki id yang sama akan dianggap sebagai data yang sama dan salah satunya perlu dihapus.

Ketiga, berdasarkan hasil pemeriksaan tipe data, sebagian besar kolom telah memiliki tipe data yang sesuai. Kolom seperti id, popularity, vote_average, dan vote_count telah menggunakan tipe data numerik, sedangkan kolom seperti adult dan softcore telah menggunakan tipe bool. Namun, kolom first_air_date masih bertipe str, padahal kolom tersebut berisi informasi tanggal. Oleh karena itu, kolom first_air_date perlu dikonversi menjadi tipe datetime agar dapat digunakan dengan tepat dalam analisis berdasarkan waktu.

### Menangani Nilai Kosong (Missing Value)

In [23]:
def bersihkan_path(teks):
    if pd.isna(teks):
        return "Tidak tersedia"
    return teks


df["backdrop_path"] = df["backdrop_path"].apply(bersihkan_path)
df["poster_path"] = df["poster_path"].apply(bersihkan_path)

print("Sel kosong setelah ditangani:")
print(df.isnull().sum())

Sel kosong setelah ditangani:
adult                0
backdrop_path        0
genre_ids            0
id                   0
origin_country       0
original_language    0
original_name        0
overview             0
popularity           0
poster_path          0
first_air_date       0
softcore             0
name                 0
vote_average         0
vote_count           0
dtype: int64


### Menangani Baris Kembar

In [24]:
jumlah_sebelum = len(df)

df_bersih = df.drop_duplicates(subset="id")

print(f"Jumlah baris sebelum : {jumlah_sebelum}")
print(f"Jumlah baris sesudah : {len(df_bersih)}")
print(f"Baris kembar dibuang : {jumlah_sebelum - len(df_bersih)}")

Jumlah baris sebelum : 1000
Jumlah baris sesudah : 933
Baris kembar dibuang : 67


### Menangani Tipe Data


In [34]:
def ubah_ke_tanggal(teks):
    return pd.to_datetime(teks)


print("Tipe data sebelum:", df_bersih["first_air_date"].dtype)

df_bersih["first_air_date"] = df_bersih["first_air_date"].apply(ubah_ke_tanggal)

print("Tipe data sesudah:", df_bersih["first_air_date"].dtype)

df_bersih.tail(10)

Tipe data sebelum: datetime64[us]
Tipe data sesudah: datetime64[us]


,adult,backdrop_path,genre_ids,id,origin_country,original_language,original_name,overview,popularity,poster_path,first_air_date,softcore,name,vote_average,vote_count
981,False,/fs5VXKvIUcfTn3nsc3g4s3OIcV4.jpg,[18],46880,[US],en,The Fosters,"Stef Foster, a dedicated police officer, and h...",47.4421,/iRCmazqaUNNsgZyR6tNDl88pXp3.jpg,2013-06-03,False,The Fosters,7.500,339
983,False,/a97Q8f3dWiaMg2nq79zG5oZVotp.jpg,"[10768, 35]",4068,[US],en,Hogan's Heroes,From a German prisoner of war camp during Worl...,47.4310,/qifpp3AmNx54Ko4obdwxOWNf0lo.jpg,1965-09-17,False,Hogan's Heroes,7.544,193
985,False,/qdVdWLUAj9Xxrx0OIT76gWVI4fg.jpg,[10764],2612,[US],en,Deadliest Catch,"Forty-foot waves, 700 pound crab pots, freezin...",47.3481,/aqF7wHfA2n89zbBqG6DLuDjrHut.jpg,2005-04-12,False,Deadliest Catch,7.243,179
987,False,/8rEKZ1PdiglSi0jTyookQtZaQOo.jpg,[99],113630,[FR],fr,Enquêtes criminelles,,47.3201,/eQNT1q6BNJt55IHk0sI1O5grUhE.jpg,2008-10-08,False,Enquêtes criminelles,8.000,1
990,False,/xIYUHL2KEYNMAsjFuUCQQZsH7D1.jpg,"[16, 35, 10762]",112527,[JP],ja,オバケのQ太郎,"Q-taro, a monster, is living with the Ohara fa...",47.2093,/acb9g0rlsFhy3LBPDvMF7Tso9Q0.jpg,1985-04-01,False,Obake no Q-tarō,8.000,3
993,False,/bVml4IQKIS5DfPqgDoaZoRLlxIY.jpg,"[16, 10751, 10762, 35, 10759]",64783,[US],en,Dawn of the Croods,The world's first family is back for more laug...,47.1301,/cyCAsMLGECvEAFDfKcxcWy7YTOW.jpg,2015-12-24,False,Dawn of the Croods,7.440,100
994,False,/mP0LywGWY77Qa9rmfWZPJDEGaY2.jpg,"[18, 10751, 35, 10759, 10766]",86310,[IN],hi,राधाकृष्ण,The story of Radha and Krishna is the epitome ...,47.0688,/3SiMmfY0663JmjpAfftmj1h6XGP.jpg,2018-10-01,False,RadhaKrishn,9.000,4
995,False,/iohe4KtMpFKYzuA3QIBRrcbJIbz.jpg,"[16, 35, 10762]",3611,[US],en,Cow and Chicken,"Follows the surreal adventures of a cow, named...",47.0546,/1vsIbN9FOsr97vIAGLYpRu7ZiSq.jpg,1997-09-16,False,Cow and Chicken,7.082,536
997,False,/32kAbClMKp2BmwNPH5j8OKzyYt9.jpg,"[18, 80]",40758,[DE],de,Großstadtrevier,Follow the everyday work of a fictional police...,47.0132,/fVsntDN3mQAuaDFGiJTKar7fRcu.jpg,1986-12-16,False,Großstadtrevier,6.000,14
998,False,/bcdbAtYOHoE2KDXjgmWxWgSJ1gP.jpg,[10767],90648,[DE],de,ZDF-Fernsehgarten,,47.0022,/kUnK5nohn8YSoG4yMWGvEKCYSuM.jpg,1986-06-29,False,ZDF-Fernsehgarten,2.800,5


In [28]:
# Mengecek kembali perubahan tipe data melalui .info()
df_bersih.info()

<class 'pandas.DataFrame'>
Index: 933 entries, 0 to 998
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   adult              933 non-null    bool          
 1   backdrop_path      933 non-null    str           
 2   genre_ids          933 non-null    object        
 3   id                 933 non-null    int64         
 4   origin_country     933 non-null    object        
 5   original_language  933 non-null    str           
 6   original_name      933 non-null    str           
 7   overview           933 non-null    str           
 8   popularity         933 non-null    float64       
 9   poster_path        933 non-null    str           
 10  first_air_date     932 non-null    datetime64[us]
 11  softcore           933 non-null    bool          
 12  name               933 non-null    str           
 13  vote_average       933 non-null    float64       
 14  vote_count         933 non

In [31]:
display(
    df_bersih[
        [
            "id",
            "name",
            "first_air_date",
            "popularity",
            "vote_average",
            "vote_count"
        ]
    ].head(10)
)

,id,name,first_air_date,popularity,vote_average,vote_count
0,275102,The Scandal,2026-09-18,3400.2893,6.273,11
1,108978,Reacher,2022-02-03,867.0873,8.100,3383
2,2261,The Tonight Show Starring Johnny Carson,1962-10-01,809.6015,7.536,96
3,63770,The Late Show with Stephen Colbert,2015-09-08,709.1087,6.082,364
4,22980,Watch What Happens Live with Andy Cohen,2009-07-16,676.0239,4.969,96
5,549,Law & Order,1990-09-13,565.5337,7.286,726
6,113962,Lioness,2023-07-23,568.1133,8.200,1592
7,62223,The Late Late Show with James Corden,2015-03-23,465.2712,5.149,148
8,59941,The Tonight Show Starring Jimmy Fallon,2014-02-17,516.0340,5.756,389
9,94722,Tagesschau,1952-12-26,461.7041,6.768,256


## **Tahap 5: Menyimpan Hasil**

Pada tahap ini kita akan menyimpan data hasil cleaning ke dalam bentik .csv

In [35]:
df_bersih.to_csv("dataset_favTV.csv", index=False)
print("Data berhasil disimpan ke file: dataset_favTV.csv")

df_cek = pd.read_csv("dataset_favTV.csv")
print(f"File terbaca kembali: {len(df_cek)} baris, {len(df_cek.columns)} kolom")
df_cek.head()

Data berhasil disimpan ke file: dataset_favTV.csv
File terbaca kembali: 933 baris, 15 kolom


,adult,backdrop_path,genre_ids,id,origin_country,original_language,original_name,overview,popularity,poster_path,first_air_date,softcore,name,vote_average,vote_count
0,False,/uc1p1PEbEMpdIHqnU9TESNC6Jhp.jpg,[18],275102,['KR'],ko,스캔들,"In Joseon, a secret proposal ignites hidden de...",3400.2893,/4TU0PcHBIl0wPW5B6aVRGExOnHE.jpg,2026-09-18,False,The Scandal,6.273,11
1,False,/pF0qkRsrHkdYadPWY9AMeFZfcwk.jpg,"[10759, 80]",108978,['US'],en,Reacher,"Jack Reacher, a veteran military police invest...",867.0873,/f1VCQIG2iCyOookdgOzwtUpwWC0.jpg,2022-02-03,False,Reacher,8.100,3383
2,False,/qFfWFwfaEHzDLWLuttWiYq7Poy2.jpg,[10767],2261,['US'],en,The Tonight Show Starring Johnny Carson,The Tonight Show Starring Johnny Carson is a t...,809.6015,/uSvET5YUvHNDIeoCpErrbSmasFb.jpg,1962-10-01,False,The Tonight Show Starring Johnny Carson,7.536,96
3,False,/gMMnf8VRg3Z98WaFmOLr9Jk8pIs.jpg,"[35, 10767]",63770,['US'],en,The Late Show with Stephen Colbert,Stephen Colbert brings his signature satire an...,709.1087,/9jkThAGYj2yp8jsS6Nriy5mzKFT.jpg,2015-09-08,False,The Late Show with Stephen Colbert,6.082,364
4,False,/hINekSpbcBxjnjGqmIm6I4bz2ab.jpg,"[10767, 35]",22980,['US'],en,Watch What Happens Live with Andy Cohen,Bravo network executive Andy Cohen discusses p...,676.0239,/onSD9UXfJwrMXWhq7UY7hGF2S1h.jpg,2009-07-16,False,Watch What Happens Live with Andy Cohen,4.969,96


## **Tahap 6 :  Bahan untuk Slide**

In [36]:
print("=" * 50)

print("ANGKA UNTUK SLIDE")

print("=" * 50)

print(f"Sumber data              : The Movie Database (TMDB)")
print(f"Jenis data               : Serial TV")
print(f"Jumlah target data       : 1000")

print()

print(f"Baris sebelum dibersihkan : {jumlah_sebelum}")
print(f"Baris dataset akhir       : {len(df_bersih)}")

print()

print("Class yang dibuat:")

print("  1. TMBDFavTV - mengambil data serial TV dari TMDB API")

print()

print("Function yang dibuat:")

print("  1. bersihkan_path - menangani nilai kosong pada path gambar")
print("  2. ubah_ke_tanggal - mengubah teks tanggal menjadi tipe datetime")

print()

print("Temuan dari pembersihan data:")

print(f"  Nilai kosong backdrop_path : 41")
print(f"  Nilai kosong poster_path   : 14")
print(f"  Baris kembar dibuang       : {jumlah_sebelum - len(df_bersih)}")
print(f"  first_air_date             : str → datetime")

print("=" * 50)

ANGKA UNTUK SLIDE
Sumber data              : The Movie Database (TMDB)
Jenis data               : Serial TV
Jumlah target data       : 1000

Baris sebelum dibersihkan : 1000
Baris dataset akhir       : 933

Class yang dibuat:
  1. TMBDFavTV - mengambil data serial TV dari TMDB API

Function yang dibuat:
  1. bersihkan_path - menangani nilai kosong pada path gambar
  2. ubah_ke_tanggal - mengubah teks tanggal menjadi tipe datetime

Temuan dari pembersihan data:
  Nilai kosong backdrop_path : 41
  Nilai kosong poster_path   : 14
  Baris kembar dibuang       : 67
  first_air_date             : str → datetime
